# 🚀 Lab 24: Interactive Plots with Plotly

### 📘 Lab Overview
In this lab, you will learn how to create **interactive visualizations using Plotly**, a modern Python library for building rich, browser-based charts. Unlike static charts, Plotly charts let users hover, zoom, pan, toggle legend items, and explore data dynamically.

You will create interactive line charts, grouped bar charts, and time-series charts with dynamic axes. You will also export plots as HTML files so they can be shared and viewed in any web browser. By the end of the lab, you will be able to build interactive dashboards and reports that go beyond traditional static visualizations.

This lab has been adapted for **Google Colab**, so everything runs directly inside notebook cells without needing a local terminal or cloud desktop.

## 🎯 Objectives
By the end of this lab, students will be able to:
* Create interactive line charts using Plotly
* Build dynamic bar charts with hover functionality
* Add legends and customize chart appearance
* Implement dynamic axes and range selectors for time-series exploration
* Export interactive charts as HTML files for sharing
* Understand the advantages of interactive visualizations over static plots
* Create multi-chart interactive dashboards for business-style reporting

## 🧰 Prerequisites
Before starting this lab, students should have:
* Basic understanding of Python programming
* Familiarity with lists and dictionaries
* Basic understanding of data visualization concepts
* Basic awareness of HTML files is helpful but not required

## ⚙️ Environment Setup

### 💡 ELI10: Setting the Stage
Before we draw cool pictures, we need to make sure our computer (Colab) has the right tools installed. We will install `plotly` for the charts, `pandas` for handling tables, and `numpy` for doing math.

In [1]:
# Install necessary libraries using pip
# %pip is the recommended way to install packages in Colab
%pip install plotly pandas numpy

# Task 1: Setting Up the Environment

## Subtask 1.3: Create Working Directory
We will create a specific folder to save our exported HTML charts.

In [2]:
import os

# Create a directory named 'plotly_lab' to store our output files
# exist_ok=True prevents an error if the folder already exists
os.makedirs("plotly_lab", exist_ok=True)
print("Working directory created successfully.")

# Move into the folder so all saved files go there automatically
%cd plotly_lab

Working directory created successfully.
/content/plotly_lab


## Subtask 1.4: Import Required Libraries
Now we tell Python to load the tools into memory.

In [3]:
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np

# Print versions to verify successful import
print("Plotly version:", plotly.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

# Set the default renderer to 'colab' to ensure charts show up in this environment
pio.renderers.default = "colab"

Plotly version: 5.24.1
Pandas version: 2.2.2
NumPy version: 2.0.2


# Task 2: Creating Interactive Line Charts

### 📈 Interactive Line Charts

### 💡 ELI10: Line Charts
Line charts are like a connect-the-dots drawing. They show how numbers change over time. With Plotly, you can hover your mouse over the dots to see the exact numbers without squinting at the axis!

In [4]:
# Sample data for monthly sales
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

sales_2022 = [15000, 18000, 22000, 25000, 28000, 32000,
              35000, 38000, 33000, 29000, 26000, 31000]

sales_2023 = [17000, 20000, 24000, 27000, 31000, 35000,
              38000, 41000, 36000, 32000, 29000, 34000]

# Create a new figure object
fig = go.Figure()

# Add the 2022 data as a line
fig.add_trace(go.Scatter(
    x=months,
    y=sales_2022,
    mode='lines+markers',
    name='Sales 2022',
    line=dict(color='blue', width=3),
    marker=dict(size=8)
))

# Add the 2023 data as a second line
fig.add_trace(go.Scatter(
    x=months,
    y=sales_2023,
    mode='lines+markers',
    name='Sales 2023',
    line=dict(color='red', width=3),
    marker=dict(size=8)
))

# Update the layout with titles and a cleaner theme
fig.update_layout(
    title='Monthly Sales Comparison',
    xaxis_title='Month',
    yaxis_title='Sales ($)',
    hovermode='x unified', # Shows all data for a specific X point when hovering
    template='plotly_white'
)

# Display the interactive chart
fig.show()

## Subtask 2.2: Adding Custom Hover Information
Instead of just showing the value, we want to show the **Growth Rate**. Since we can't do complex math inside the hover string, we precalculate it in Python and pass it to Plotly.

In [5]:
# Precompute Year-over-Year growth percentages
growth_rates = [((s23 - s22) / s22) * 100 for s22, s23 in zip(sales_2022, sales_2023)]

fig_enhanced = go.Figure()

# 2022 Trace
fig_enhanced.add_trace(go.Scatter(
    x=months, y=sales_2022, mode='lines+markers', name='Sales 2022',
    line=dict(color='blue', width=3),
    hovertemplate='<b>%{fullData.name}</b><br>Month: %{x}<br>Sales: $%{y:,.0f}<extra></extra>'
))

# 2023 Trace with precomputed growth data passed via 'customdata'
fig_enhanced.add_trace(go.Scatter(
    x=months, y=sales_2023, mode='lines+markers', name='Sales 2023',
    line=dict(color='red', width=3),
    customdata=growth_rates, # This passes our list to the chart
    hovertemplate='<b>%{fullData.name}</b><br>Month: %{x}<br>Sales: $%{y:,.0f}<br>Growth vs 2022: %{customdata:.1f}%<extra></extra>'
))

fig_enhanced.update_layout(
    title={'text': 'Interactive Monthly Sales Comparison', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Month', yaxis_title='Sales ($)',
    hovermode='x unified', template='plotly_white'
)

fig_enhanced.show()

# Task 3: Building Interactive Bar Charts

### 📊 Interactive Bar Charts

### 💡 ELI10: Bar Charts
Bar charts use tall or short blocks to compare different groups. Here, we compare sales for Laptops, Phones, and more. Interactive bars let us see what percentage of the total each product represents just by looking at it!

In [6]:
# Product data
products = ['Laptops', 'Smartphones', 'Tablets', 'Headphones', 'Cameras']
q1_sales = [45000, 62000, 28000, 15000, 22000]
q2_sales = [52000, 68000, 32000, 18000, 25000]
q3_sales = [48000, 71000, 30000, 20000, 28000]

# Totals for percentage calculation
total_q1, total_q2, total_q3 = sum(q1_sales), sum(q2_sales), sum(q3_sales)

fig_bar_enhanced = go.Figure()

# Helper to add bars with custom percentage hover
def add_quarter_bar(name, y_data, total, color):
    fig_bar_enhanced.add_trace(go.Bar(
        name=name, x=products, y=y_data, marker_color=color,
        customdata=[val / total * 100 for val in y_data],
        hovertemplate='<b>%{x}</b><br>Quarter: '+name+'<br>Sales: $%{y:,.0f}<br>Percentage of Total: %{customdata:.1f}%<extra></extra>'
    ))

add_quarter_bar('Q1 2023', q1_sales, total_q1, 'lightblue')
add_quarter_bar('Q2 2023', q2_sales, total_q2, 'darkblue')
add_quarter_bar('Q3 2023', q3_sales, total_q3, 'navy')

fig_bar_enhanced.update_layout(
    title={'text': 'Interactive Quarterly Product Sales Dashboard', 'x': 0.5, 'xanchor': 'center'},
    xaxis_title='Products', yaxis_title='Sales ($)', barmode='group',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white'
)

fig_bar_enhanced.show()

# Task 4: Adding Legends and Dynamic Axes

### 🧭 Legends and Dynamic Axes

### 💡 ELI10: Dynamic Axes
Sometimes we have too much data to see at once (like a whole year of stock prices). Dynamic axes let us add a "zoom slider" at the bottom and buttons to jump to specific times like "last 7 days" or "last month."

In [7]:
# Generate 365 days of dates
dates = pd.date_range(start='2023-01-01', periods=365, freq='D')

# Simulate a random walk for stock prices
np.random.seed(42)
price_changes = np.random.normal(0, 2, len(dates))
stock_prices = np.cumsum(price_changes) + 100 # Start at $100

fig_dynamic = go.Figure()
fig_dynamic.add_trace(go.Scatter(x=dates, y=stock_prices, mode='lines', name='Stock Price', line=dict(color='blue')))

# Configure the range selector (buttons) and range slider
fig_dynamic.update_layout(
    title='Stock Price with Dynamic Axes',
    xaxis=dict(
        title='Date',
        rangeselector=dict(
            buttons=list([
                dict(count=7, label='7d', step='day', stepmode='backward'),
                dict(count=30, label='30d', step='day', stepmode='backward'),
                dict(count=90, label='3m', step='day', stepmode='backward'),
                dict(step='all', label='All')
            ])
        ),
        rangeslider=dict(visible=True), # Adds the zoom bar at the bottom
        type='date'
    ),
    yaxis=dict(title='Price ($)', fixedrange=False),
    template='plotly_white'
)

fig_dynamic.show()

# Task 5: Exporting Interactive Plots as HTML

### 💾 Exporting HTML Files

### 💡 ELI10: Saving your work
One of the best parts of Plotly is that you can save these charts as `.html` files. You can email these files to anyone, and they can open them in Chrome or Safari and still use all the interactive features without needing Python!

In [8]:
# Export individual charts
fig_enhanced.write_html("sales_comparison.html")
fig_bar_enhanced.write_html("quarterly_sales.html")
fig_dynamic.write_html("stock_price_dynamic.html")

print("Individual HTML files exported successfully.")

Individual HTML files exported successfully.


## Subtask 5.3: Creating a Complete HTML Report
We will now combine multiple charts into one single professional web report.

In [9]:
import json

# Define the HTML structure as a multi-line string
# We use double curly braces {{ }} for CSS so .format() doesn't get confused
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Sales Analytics Report</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 40px; background-color: #f4f7f6; }}
        .container {{ max-width: 1000px; margin: auto; background: white; padding: 20px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); }}
        h1 {{ color: #2c3e50; text-align: center; }}
        .chart-card {{ margin: 30px 0; padding: 20px; border: 1px solid #eee; border-radius: 8px; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Business Intelligence Report</h1>
        <div class="chart-card">
            <h2>Monthly Trends</h2>
            <div id="chart1"></div>
        </div>
        <div class="chart-card">
            <h2>Product Performance</h2>
            <div id="chart2"></div>
        </div>
    </div>
    <script>
        var data1 = {json1};
        Plotly.newPlot('chart1', data1.data, data1.layout);
        var data2 = {json2};
        Plotly.newPlot('chart2', data2.data, data2.layout);
    </script>
</body>
</html>
"""

# Convert Plotly figures to JSON strings
json_str1 = fig_enhanced.to_json()
json_str2 = fig_bar_enhanced.to_json()

# Fill the template
final_report = html_template.format(json1=json_str1, json2=json_str2)

# Save the file
with open("complete_sales_report.html", "w", encoding="utf-8") as f:
    f.write(final_report)

print("Complete HTML report 'complete_sales_report.html' created.")

Complete HTML report 'complete_sales_report.html' created.


# Task 6: Verification and Testing

## ✅ Verification Checklist
* [ ] Libraries imported without error
* [ ] Line chart shows hover growth rates correctly
* [ ] Bar charts are grouped and show percentages
* [ ] Stock chart has a working range slider
* [ ] HTML files exist in the file browser (left sidebar)

## Subtask 6.1: Verify Files Exist

In [10]:
import glob
# Check the directory for all HTML files
files_found = glob.glob("*.html")
print(f"Verification: Found {len(files_found)} HTML files.")
for f in files_found:
    size = os.path.getsize(f) / 1024
    print(f"- {f} ({size:.2f} KB)")

Verification: Found 4 HTML files.
- stock_price_dynamic.html (4474.31 KB)
- quarterly_sales.html (4461.16 KB)
- sales_comparison.html (4460.74 KB)
- complete_sales_report.html (17.65 KB)


## 🛠 Troubleshooting
1. **Chart not appearing?** Ensure `pio.renderers.default = "colab"` is set.
2. **Hover text broken?** Remember that logic like `if/else` doesn't work inside `hovertemplate`. Always precompute in Python.
3. **HTML file empty?** Check that you used `.write_html()` on the figure object.

## 📚 Key Takeaways
- **Plotly** is browser-based, making it superior for sharing interactive data.
- **Customdata** is the secret to rich, informative hover labels.
- **Dashboards** can be created simply by combining JSON outputs into an HTML template.

## 🎓 Conclusion
You've moved from static charts to dynamic data experiences! These skills allow you to build reports that let stakeholders find their own answers by interacting with the visuals.

### What I Learned
I learned how to use Plotly's `graph_objects` to build complex layouts, manage custom hover data, and export everything into standalone web reports.

### Real-World Applications
- **Finance**: Tracking stock volatility with range sliders.
- **Marketing**: Comparing campaign ROI across regions using grouped bars.
- **Operations**: Monitoring monthly growth KPIs.